# About the Notebook

Inference Backends: Cold Starts, Caching, and Deployment Physics

When your agent calls a model, it is not talking to "an API." It is talking to a
**GPU process** — and that process has physics. Models must be loaded into VRAM before
the first token can be generated. KV caches must be computed for every new prefix.
Memory must be allocated, managed, and sometimes evicted.

These are the deployment realities that determine whether your agent responds in
200ms or 12 seconds:

| Concern | What Happens | Impact on Your Agent |
|---|---|---|
| **Cold Start** | Model weights loaded from disk/network into GPU VRAM | First request after deploy takes 30–120s instead of <1s |
| **KV Cache** | Attention keys/values computed for every token in the prompt | Long system prompts re-computed on every call unless cached |
| **Prefix Caching** | Reuse KV cache for shared prompt prefixes across requests | Same system prompt → skip recomputation → 2-5× TTFT improvement |
| **Cache Invalidation** | Prefix changes → cached KV entries become useless | New system prompt or conversation fork = cold-cache latency spike |
| **Memory Pressure** | KV cache grows with context length × batch size | Long conversations evict other requests' caches, spiking tail latency |

This notebook starts a local **vLLM** inference server, then measures each of these
effects with real requests — so you can see the numbers, not just read about them.

We also compare the three major open-source inference backends — **vLLM**, **TGI**,
and **SGLang** — so you know when to reach for which.



**Cold starts are a deployment tax, not a bug.**
Every time you redeploy, scale from zero, or restart after a crash, there's a 30-120s
window where your agent is unresponsive. Mitigate with keep-alive probes, pre-warming
requests, or min-replica > 0 scaling policies.

**The KV cache is your agent's working memory — and it's expensive.**
Longer conversations = more VRAM = slower TTFT = fewer concurrent users. This is
why context pruning (previous notebook) isn't just a cost optimization — it's a
latency optimization.

**Prefix caching is nearly free and dramatically reduces TTFT.**
If your agent uses a stable system prompt (and it should), enable prefix caching.
But be aware that dynamic prefix content (RAG snippets, user profiles) invalidates
the cache. Design your prompt template with a **stable prefix** and **variable suffix**.

**Cache invalidation is the hidden cost of flexibility.**
Every time you change the system prompt, fork a conversation, or inject new context
at the top of the prompt, you pay the full KV computation cost again. This is the
trade-off between agent flexibility and inference efficiency.

**Your agent code should be backend-agnostic.**
Use the OpenAI SDK as your client library. vLLM, TGI, SGLang, and every cloud
provider speak the same API. Swapping backends is a config change, not a refactor.

```
Production Deployment Checklist:
  ☐  Inference backend selected (vLLM / TGI / SGLang)
  ☐  Prefix caching enabled
  ☐  System prompt designed with stable prefix
  ☐  Cold start measured and documented
  ☐  Health check endpoint monitored
  ☐  Min replicas > 0 (avoid scale-to-zero cold starts)
  ☐  Context pruning configured (max history tokens)
  ☐  GPU memory budget estimated (model + KV cache + overhead)
```

# Inference Backends: vLLM vs. TGI vs. SGLang

All three are open-source, GPU-native inference servers that expose OpenAI-compatible
APIs. Your agent code doesn't change — `client.chat.completions.create()` works with
all of them. What changes is the **operational profile**.

| Feature | **vLLM** | **TGI** (Text Generation Inference) | **SGLang** |
|---|---|---|---|
| **Maintained by** | UC Berkeley (open-source) | Hugging Face | LMSYS |
| **Scheduling** | PagedAttention (memory-efficient KV) | Continuous batching | RadixAttention (tree-based KV reuse) |
| **Prefix Caching** | `--enable-prefix-caching` (APC) | ✗ (manual prompt caching only) | Built-in (RadixAttention) |
| **Multi-GPU** | Tensor parallel + pipeline parallel | Tensor parallel | Tensor parallel + expert parallel |
| **Speculative Decoding** | ✓ (draft model or n-gram) | ✓ (medusa heads) | ✓ |
| **Quantization** | AWQ, GPTQ, FP8, BitsAndBytes | AWQ, GPTQ, EETQ, BitsAndBytes | AWQ, GPTQ, FP8 |
| **Structured Output** | ✓ (guided decoding / outlines) | ✓ (grammar-based) | ✓ (compressed FSM) |
| **Vision Models** | ✓ | ✓ | ✓ |
| **Best For** | General-purpose, high throughput | HF ecosystem, quick deploy | Multi-turn agents (tree caching) |
| **Cold Start** | ~30-120s (model dependent) | ~20-90s | ~30-120s |
| **API Compat** | OpenAI-compatible | OpenAI-compatible + TGI native | OpenAI-compatible + SGLang native |

### When to use which:

- **vLLM**: Default choice. Best documentation, widest model support, battle-tested
  PagedAttention. Use this unless you have a specific reason not to.

- **TGI**: You're already in the Hugging Face ecosystem (Inference Endpoints, SageMaker).
  Tight integration with `transformers` and HF Hub. Easiest path to production if you're
  already a HF shop.

- **SGLang**: Your agent does heavy multi-turn reasoning (tree-of-thought, self-consistency,
  branching tool calls). RadixAttention reuses KV cache across conversation branches
  more efficiently than vLLM's linear prefix caching. Also excellent for structured
  output with its compressed finite-state machine approach.

In [ ]:
%pip install -q vllm openai

In [ ]:
import json, os, time, subprocess, signal, statistics
from pathlib import Path
from openai import OpenAI
import requests

# ── Model configuration ───────────────────────────────────────────────
# Swap the model for anything your GPU can hold.
# Qwen2.5-3B fits on a single T4 (16 GB). For an A100, try 8B or 14B.
VLLM_MODEL       = "Qwen/Qwen2.5-3B-Instruct"
SERVED_NAME       = "qwen"

# ── Engine args ───────────────────────────────────────────────────────
# These map directly to vLLM's engine arguments:
#   https://docs.vllm.ai/en/stable/configuration/engine_args/
VLLM_PORT              = 8001
VLLM_HOST              = "0.0.0.0"
TENSOR_PARALLEL_SIZE   = 1        # number of GPUs for tensor parallelism
GPU_MEMORY_UTILIZATION = 0.90     # fraction of GPU VRAM vLLM may use
MAX_MODEL_LEN          = 4096     # max context window (tokens)
DTYPE                  = "auto"   # "auto" | "float16" | "bfloat16"
SWAP_SPACE             = 4        # GiB of CPU swap for KV cache overflow
MAX_NUM_SEQS           = 64       # max concurrent sequences in a batch
DISABLE_LOG_STATS      = True     # quieter logs in notebook

# The OpenAI-compatible client — same interface you already use for
# OpenRouter, Azure, or any hosted API. That's the point: your agent
# code doesn't change when you swap the backend.
local_client = OpenAI(
    base_url=f"http://localhost:{VLLM_PORT}/v1",
    api_key="not-needed",  # local server, no auth
)

print(f"Model:  {VLLM_MODEL}")
print(f"Server: http://localhost:{VLLM_PORT}/v1")
print(f"Engine: TP={TENSOR_PARALLEL_SIZE}, GPU mem={GPU_MEMORY_UTILIZATION}, "
      f"max_len={MAX_MODEL_LEN}, dtype={DTYPE}")

Model:  Qwen/Qwen2.5-3B-Instruct
Server: http://localhost:8001/v1
Engine: TP=1, GPU mem=0.9, max_len=4096, dtype=auto


## 1 — Cold Start: From `vllm serve` to First Token

The cold start isn't just "the server takes a while to boot." It's a **chain of
blocking steps**, each of which adds latency before your agent can serve its first
user:

```
 ┌────────────────┐   ┌──────────────────┐   ┌──────────────────────┐   ┌─────────────┐
 │ Download model │ → │ Load into VRAM   │ → │ 1st request: alloc   │ → │ Warm request │
 │ weights (if    │   │ (deserialize +   │   │ KV cache, compile    │   │ (steady      │
 │ not cached)    │   │ shard across     │   │ remaining kernels,   │   │  state)      │
 │                │   │ GPUs)            │   │ warm scheduler       │   │              │
 │ ~0-60s         │   │ ~15-60s          │   │ ~1-5s extra          │   │ ~0.2-1s      │
 └────────────────┘   └──────────────────┘   └──────────────────────┘   └─────────────┘
     Phase 1                Phase 2                  Phase 3                Phase 4
               ──────── cold_start_seconds ────────  ─ 1st req ─  ── warm ──
```

We measure all of this end-to-end: start the server from scratch, wait for healthy,
then time the first few requests to see the full cold-to-warm transition.

We launch vLLM via `python -m vllm.entrypoints.openai.api_server` — the **OpenAI-compatible**
entrypoint that exposes `/v1/chat/completions` (same as `vllm serve` under the hood).
The flags map directly to vLLM's **[Engine Arguments](https://docs.vllm.ai/en/stable/configuration/engine_args/)**:

| Flag | What it controls | Why it matters |
|---|---|---|
| `--model` | HF model ID or local path | What gets loaded into GPU VRAM |
| `--tensor-parallel-size` | Number of GPUs for TP | Splits the model across GPUs |
| `--gpu-memory-utilization` | Fraction of VRAM for KV cache | Too high → OOM; too low → fewer concurrent seqs |
| `--max-model-len` | Max context window | Caps KV cache memory per request |
| `--swap-space` | CPU RAM (GiB) for KV overflow | Safety valve when GPU cache is full |
| `--dtype` | Compute precision | `auto` picks bf16/fp16 based on GPU |
| `--max-num-seqs` | Max concurrent sequences | Bounds memory & controls batching |
| `--enforce-eager` | Disable CUDA graph capture | Faster startup, slightly lower throughput |
| `--disable-log-stats` | Suppress periodic stats | Cleaner logs for notebooks |

In [ ]:
import subprocess, time, requests

VLLM_LOG = "vllm_server.log"


def build_vllm_args(
    model: str = VLLM_MODEL,
    served_name: str = SERVED_NAME,
    port: int = VLLM_PORT,
    extra_args: list[str] | None = None,
) -> list[str]:
    """Build the vLLM server command using python -m vllm.entrypoints.openai.api_server.

    Every flag here maps to a vLLM Engine Argument:
    https://docs.vllm.ai/en/stable/configuration/engine_args/
    """
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        f"--host={VLLM_HOST}",
        f"--port={port}",
        f"--model={model}",
        f"--served-model-name={served_name}",
        f"--tensor-parallel-size={TENSOR_PARALLEL_SIZE}",
        f"--gpu-memory-utilization={GPU_MEMORY_UTILIZATION}",
        f"--max-model-len={MAX_MODEL_LEN}",
        f"--dtype={DTYPE}",
        f"--swap-space={SWAP_SPACE}",
        f"--max-num-seqs={MAX_NUM_SEQS}",
        "--enforce-eager",          # faster startup, skip CUDA graph capture
    ]
    if DISABLE_LOG_STATS:
        cmd.append("--disable-log-stats")
    if extra_args:
        cmd.extend(extra_args)
    return cmd


def start_vllm_server(
    model: str = VLLM_MODEL,
    served_name: str = SERVED_NAME,
    port: int = VLLM_PORT,
    extra_args: list[str] | None = None,
) -> subprocess.Popen:
    """Start a vLLM server process and return the handle."""
    cmd = build_vllm_args(model, served_name, port, extra_args)

    log_fh = open(VLLM_LOG, "w")
    proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

    # Pretty-print the launch command
    print(f"  vLLM PID {proc.pid}")
    print(f"  cmd: \\\n    " + " \\\n    ".join(cmd))
    return proc


def wait_for_healthy(port: int = VLLM_PORT, timeout: int = 300) -> float:
    """Block until the vLLM health endpoint responds. Returns seconds waited."""
    url = f"http://localhost:{port}/health"
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            r = requests.get(url, timeout=2)
            if r.status_code == 200:
                elapsed = time.time() - t0
                print(f"  ✅ Server healthy after {elapsed:.1f}s")
                return elapsed
        except requests.ConnectionError:
            pass
        time.sleep(2)
    raise TimeoutError(f"vLLM not healthy after {timeout}s — check {VLLM_LOG}")


def stop_vllm_server(proc: subprocess.Popen):
    """Gracefully stop the vLLM server."""
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
    print(f"  Server stopped (PID {proc.pid})")

In [ ]:
SYSTEM_PROMPT = (
    "You are a customer support triage agent for an e-commerce company. "
    "Classify tickets by category and priority. Be concise."
)

TEST_MESSAGE = "I placed order #ORD-8842 three weeks ago and nobody has told me anything. This is unacceptable."


def timed_completion(
    client: OpenAI,
    messages: list[dict],
    model: str = SERVED_NAME,
    max_tokens: int = 256,
) -> dict:
    """Send a chat completion and return timing + usage metadata."""
    t0 = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.2,
    )
    elapsed = time.time() - t0

    usage = response.usage
    content = response.choices[0].message.content or ""

    return {
        "latency_s": round(elapsed, 3),
        "prompt_tokens": usage.prompt_tokens if usage else 0,
        "completion_tokens": usage.completion_tokens if usage else 0,
        "total_tokens": usage.total_tokens if usage else 0,
        "content": content,
        "tokens_per_sec": round(
            (usage.completion_tokens / elapsed) if usage and elapsed > 0 else 0, 1
        ),
    }



#  PHASE 1+2: Start server from scratch, measure boot time

print("=" * 70)
print(" COLD START — FULL END-TO-END MEASUREMENT")
print("=" * 70)

t_total_start = time.time()

print("\n  Phase 1+2: Starting vLLM server (model download + GPU loading)...\n")
vllm_proc = start_vllm_server()
cold_start_seconds = wait_for_healthy()

print(f"\n  ⏱  Server boot (Phase 1+2): {cold_start_seconds:.1f}s")
print(f"     Model loaded into VRAM, health endpoint responding.\n")


#  PHASE 3+4: First requests — cold GPU → warm steady state

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": TEST_MESSAGE},
]

results = []
labels = [
    "1st request (cold GPU)  ← Phase 3",
    "2nd request (warming)",
    "3rd request (warm)      ← Phase 4",
    "4th request (warm)",
    "5th request (warm)",
]

print("  Phase 3+4: Request latencies after boot:")
print("  " + "─" * 66)

for i, label in enumerate(labels):
    r = timed_completion(local_client, messages)
    results.append(r)
    print(
        f"  {label:<35} │ {r['latency_s']:>6.3f}s │ "
        f"{r['completion_tokens']:>4} tok │ "
        f"{r['tokens_per_sec']:>6.1f} tok/s"
    )

t_total_first_response = time.time() - t_total_start

# Summary
cold_req      = results[0]["latency_s"]
warm_avg      = statistics.mean(r["latency_s"] for r in results[2:])  # 3rd-5th
total_to_first = cold_start_seconds + cold_req

print(f"\n{'═' * 70}")
print(f"  COLD START BREAKDOWN")
print(f"{'─' * 70}")
print(f"  Server boot (Phase 1+2):   {cold_start_seconds:>7.1f}s   model → VRAM")
print(f"  1st request  (Phase 3):    {cold_req:>7.3f}s   KV alloc + kernel compile")
print(f"  ────────────────────────── ─────────")
print(f"  Total cold start:          {total_to_first:>7.1f}s   deploy → first answer")
print(f"")
print(f"  Warm request (Phase 4):    {warm_avg:>7.3f}s   steady state")
print(f"  Cold/Warm ratio:           {total_to_first / warm_avg:>7.0f}×   ← your user waits this much longer")
print(f"{'─' * 70}")
print(f"  → After every deploy, restart, or scale-from-zero event, your agent")
print(f"    is OFFLINE for ~{total_to_first:.0f}s. Plan for it: health checks, warm-up")
print(f"    probes, min-replicas > 0, or rolling deploys.")

 COLD START — FULL END-TO-END MEASUREMENT

  Phase 1+2: Starting vLLM server (model download + GPU loading)...

  vLLM PID 3941
  cmd: \
    python \
    -m \
    vllm.entrypoints.openai.api_server \
    --host=0.0.0.0 \
    --port=8001 \
    --model=Qwen/Qwen2.5-3B-Instruct \
    --served-model-name=qwen \
    --tensor-parallel-size=1 \
    --gpu-memory-utilization=0.9 \
    --max-model-len=4096 \
    --dtype=auto \
    --swap-space=4 \
    --max-num-seqs=64 \
    --enforce-eager \
    --disable-log-stats
  ✅ Server healthy after 124.1s

  ⏱  Server boot (Phase 1+2): 124.1s
     Model loaded into VRAM, health endpoint responding.

  Phase 3+4: Request latencies after boot:
  ──────────────────────────────────────────────────────────────────
  1st request (cold GPU)  ← Phase 3   │  0.458s │   10 tok │   21.9 tok/s
  2nd request (warming)               │  0.276s │   10 tok │   36.2 tok/s
  3rd request (warm)      ← Phase 4   │  0.277s │   10 tok │   36.1 tok/s
  4th request (warm)      

# Context Length and KV Cache Pressure

> **Note:** This section applies specifically to **decoder-only** (autoregressive)
> models — GPT, LLaMA, Qwen, Mistral, etc. — which is what vLLM, TGI, and SGLang
> serve. Encoder-decoder architectures (T5, BART) compute the encoder KV cache once
> and reuse it across all decode steps, so prompt length has a much smaller impact
> on per-token generation latency there.

In decoder-only models, every token in the prompt produces a KV pair that must be
stored in GPU memory **and** attended to on every subsequent generation step. As your
agent's conversation history grows, three things happen:

1. **TTFT (time-to-first-token) increases** — the prefill pass must compute attention
   over the entire prompt; cost scales quadratically with sequence length
2. **Memory pressure rises** — KV cache grows linearly with context length, leaving
   less room for batching concurrent requests
3. **Per-token decode slows down** — each new token attends to all previous KV pairs,
   so longer contexts mean more memory reads per step

This is why context pruning (from the previous notebook) isn't just about cost —
it's about keeping your inference backend responsive.

In [ ]:
# ── Build prompts of increasing context length ───────────────────────
FILLER_TURN = (
    "The customer previously wrote: 'I ordered a blue widget on January 5th "
    "and the tracking says it's stuck in Memphis. Can someone look into this? "
    "I also have a question about your return policy for items over $50.'"
)

def build_messages(num_history_turns: int) -> list[dict]:
    """Build a conversation with N filler turns + the real question."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for i in range(num_history_turns):
        msgs.append({"role": "user", "content": f"[Turn {i+1}] {FILLER_TURN}"})
        msgs.append({"role": "assistant", "content": f"Acknowledged turn {i+1}."})
    msgs.append({"role": "user", "content": TEST_MESSAGE})
    return msgs


# ── Measure latency vs context size ──────────────────────────────────
turn_counts = [0, 5, 10, 20, 30]
context_results = []

print("=" * 70)
print(" CONTEXT LENGTH vs. LATENCY")
print("=" * 70)

for n in turn_counts:
    msgs = build_messages(n)
    r = timed_completion(local_client, msgs, max_tokens=128)
    context_results.append({"turns": n, **r})
    print(
        f"  {n:>3} history turns │ {r['prompt_tokens']:>5} prompt tok │ "
        f"{r['latency_s']:>6.3f}s │ {r['tokens_per_sec']:>6.1f} tok/s"
    )

# ── Compute scaling factor ────────────────────────────────────────────
baseline = context_results[0]["latency_s"]
heaviest = context_results[-1]["latency_s"]
print(f"\n{'─' * 70}")
print(f"  Baseline (0 turns):   {baseline:.3f}s")
print(f"  Heaviest ({turn_counts[-1]} turns): {heaviest:.3f}s")
print(f"  Slowdown:             {heaviest / baseline:.1f}×")
print(f"\n  → Long agent conversations get slower even if the final question is identical.")

 CONTEXT LENGTH vs. LATENCY
    0 history turns │    62 prompt tok │  0.282s │   35.4 tok/s
    5 history turns │   407 prompt tok │  0.223s │   31.4 tok/s
   10 history turns │   754 prompt tok │  0.286s │   34.9 tok/s
   20 history turns │  1464 prompt tok │  0.232s │   30.2 tok/s
   30 history turns │  2174 prompt tok │  0.309s │   32.4 tok/s

──────────────────────────────────────────────────────────────────────
  Baseline (0 turns):   0.282s
  Heaviest (30 turns): 0.309s
  Slowdown:             1.1×

  → Long agent conversations get slower even if the final question is identical.


# Prefix Caching: A/B Comparison

Most agents use the **same system prompt** for every request. Without prefix caching,
the server recomputes the KV cache for that system prompt on every single call —
pure waste.

vLLM's `--enable-prefix-caching` (APC — Automatic Prefix Caching) detects when
multiple requests share a common prefix and reuses the cached KV blocks:

```
Request 1:  [SYSTEM: "You are a support agent..."] + [USER: "My order is late"]
                    ↑ compute KV, store in cache

Request 2:  [SYSTEM: "You are a support agent..."] + [USER: "Refund policy?"]
                    ↑ cache HIT → skip KV computation for this prefix
```

To see the real difference, we run the **exact same 5 questions** twice:
1. **Without** prefix caching (current server)
2. **With** `--enable-prefix-caching` (restart server)

Then compare side by side.

In [ ]:
#  5 test questions (same system prompt, different user messages) ─
user_messages = [
    "I placed order #ORD-8842 three weeks ago and nobody has told me anything.",
    "What is your return policy for electronics?",
    "I need to cancel order #ORD-1234 immediately.",
    "My package arrived damaged — the box was crushed.",
    "Can I change the shipping address on my pending order?",
]


def run_prefix_experiment(client, label: str) -> list[dict]:
    """Send the same 5 questions and collect latency results."""
    # Warm-up request (not counted) — primes GPU scheduler / KV allocator
    _ = timed_completion(client, messages, max_tokens=16)
    time.sleep(0.5)

    results = []
    for i, user_msg in enumerate(user_messages):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        r = timed_completion(client, msgs, max_tokens=128)
        results.append(r)
        print(
            f"  Q{i+1}: {r['latency_s']:>6.3f}s │ "
            f"{r['prompt_tokens']:>4} prompt tok │ "
            f"{r['tokens_per_sec']:>6.1f} tok/s │ "
            f"{user_msg[:50]}..."
        )
    return results



#  RUN A: WITHOUT prefix caching (current server)

print("=" * 70)
print(" RUN A — WITHOUT prefix caching")
print("=" * 70)
results_no_cache = run_prefix_experiment(local_client, "no-cache")


#  Restart server WITH prefix caching
print(f"\n{'─' * 70}")
print("Stopping current server...")
stop_vllm_server(vllm_proc)
time.sleep(3)

print("\nStarting vLLM server WITH --enable-prefix-caching...")
vllm_proc = start_vllm_server(extra_args=["--enable-prefix-caching"])
wait_for_healthy()


#  RUN B: WITH prefix caching

print("\n" + "=" * 70)
print(" RUN B — WITH prefix caching")
print("=" * 70)
results_with_cache = run_prefix_experiment(local_client, "with-cache")


#  SIDE-BY-SIDE COMPARISON

print("\n" + "=" * 70)
print(" A/B COMPARISON — prefix caching OFF vs ON")
print("=" * 70)
print(f"  {'Q':<3} │ {'Without':>9} │ {'With':>9} │ {'Speedup':>8} │ Question")
print(f"  {'─'*3}─┼─{'─'*9}─┼─{'─'*9}─┼─{'─'*8}─┼─{'─'*40}")

for i, (no_c, with_c) in enumerate(zip(results_no_cache, results_with_cache)):
    speedup = no_c["latency_s"] / with_c["latency_s"] if with_c["latency_s"] > 0 else 0
    print(
        f"  Q{i+1}  │ {no_c['latency_s']:>8.3f}s │ {with_c['latency_s']:>8.3f}s │ "
        f"{speedup:>7.2f}× │ {user_messages[i][:40]}..."
    )

avg_no   = statistics.mean(r["latency_s"] for r in results_no_cache)
avg_with = statistics.mean(r["latency_s"] for r in results_with_cache)
avg_speedup = avg_no / avg_with if avg_with > 0 else 0

# Use results from Q2-Q5 for "cached" comparison (Q1 primes the prefix cache)
avg_no_2_5   = statistics.mean(r["latency_s"] for r in results_no_cache[1:])
avg_with_2_5 = statistics.mean(r["latency_s"] for r in results_with_cache[1:])
cached_speedup = avg_no_2_5 / avg_with_2_5 if avg_with_2_5 > 0 else 0

print(f"\n{'─' * 70}")
print(f"  Avg (all 5):   {avg_no:.3f}s → {avg_with:.3f}s  ({avg_speedup:.2f}× speedup)")
print(f"  Avg (Q2-Q5):   {avg_no_2_5:.3f}s → {avg_with_2_5:.3f}s  ({cached_speedup:.2f}× speedup)")
print(f"\n  → Q1 primes the prefix cache. From Q2 onward the system prompt KV is")
print(f"    reused — that's the steady-state benefit your agent gets in production.")

 RUN A — WITHOUT prefix caching
  Q1:  0.304s │   58 prompt tok │   32.9 tok/s │ I placed order #ORD-8842 three weeks ago and nobod...
  Q2:  1.033s │   46 prompt tok │   35.8 tok/s │ What is your return policy for electronics?...
  Q3:  0.228s │   52 prompt tok │   30.6 tok/s │ I need to cancel order #ORD-1234 immediately....
  Q4:  0.282s │   48 prompt tok │   35.4 tok/s │ My package arrived damaged — the box was crushed....
  Q5:  0.258s │   49 prompt tok │   34.9 tok/s │ Can I change the shipping address on my pending or...

──────────────────────────────────────────────────────────────────────
Stopping current server...
  Server stopped (PID 3941)

Starting vLLM server WITH --enable-prefix-caching...
  vLLM PID 4853
  cmd: \
    python \
    -m \
    vllm.entrypoints.openai.api_server \
    --host=0.0.0.0 \
    --port=8001 \
    --model=Qwen/Qwen2.5-3B-Instruct \
    --served-model-name=qwen \
    --tensor-parallel-size=1 \
    --gpu-memory-utilization=0.9 \
    --max-model-len=40

# Cache Invalidation: When the Prefix Changes

Prefix caching is powerful — until the prefix changes. In agent systems, this happens
when you:

- **Change the system prompt** (A/B testing, role switching)
- **Inject dynamic context** (RAG retrieval, user profile, tool results)
- **Fork a conversation** (branching agent paths)

Each prefix change is a **cache miss** — the server must recompute the KV cache from
scratch for the new prefix. This is the "cache invalidation tax."

In [ ]:
# ── Different system prompts = cache misses ───────────────────────────
SYSTEM_PROMPTS = {
    "support_agent": (
        "You are a customer support triage agent for an e-commerce company. "
        "Classify tickets by category and priority. Be concise."
    ),
    "returns_specialist": (
        "You are a returns and refund specialist. Help customers with return "
        "eligibility, refund timelines, and exchange options. Be precise about "
        "policy details and timelines."
    ),
    "escalation_agent": (
        "You are an escalation manager who handles VIP customers and complex "
        "cases. Use a formal, empathetic tone. Offer concrete next steps "
        "and set clear expectations for resolution timelines."
    ),
    "back_to_support": (
        "You are a customer support triage agent for an e-commerce company. "
        "Classify tickets by category and priority. Be concise."
    ),
}

fixed_user_msg = "My order hasn't arrived and I'm really frustrated."

print("=" * 70)
print(" CACHE INVALIDATION — CHANGING THE SYSTEM PROMPT")
print("=" * 70)

invalidation_results = {}
for label, sys_prompt in SYSTEM_PROMPTS.items():
    msgs = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": fixed_user_msg},
    ]
    r = timed_completion(local_client, msgs, max_tokens=128)
    invalidation_results[label] = r
    cache_status = "MISS" if label != "back_to_support" else "HIT (same as #1)"
    print(
        f"  {label:<22} │ {r['latency_s']:>6.3f}s │ "
        f"{r['tokens_per_sec']:>6.1f} tok/s │ cache: {cache_status}"
    )

# ── Highlight the cost of switching ──────────────────────────────────
first = invalidation_results["support_agent"]["latency_s"]
back  = invalidation_results["back_to_support"]["latency_s"]
print(f"\n{'─' * 70}")
print(f"  First 'support_agent':    {first:.3f}s  (cache miss → compute KV)")
print(f"  Return to same prompt:    {back:.3f}s   (cache hit → reuse KV)")
print(f"\n  → Dynamic system prompts (RAG injection, role switching) invalidate")
print(f"    the prefix cache. Design for stable prefixes where possible.")

 CACHE INVALIDATION — CHANGING THE SYSTEM PROMPT
  support_agent          │  0.283s │   35.3 tok/s │ cache: MISS
  returns_specialist     │  3.253s │   39.4 tok/s │ cache: MISS
  escalation_agent       │  3.267s │   39.2 tok/s │ cache: MISS
  back_to_support        │  0.315s │   34.9 tok/s │ cache: HIT (same as #1)

──────────────────────────────────────────────────────────────────────
  First 'support_agent':    0.283s  (cache miss → compute KV)
  Return to same prompt:    0.315s   (cache hit → reuse KV)

  → Dynamic system prompts (RAG injection, role switching) invalidate
    the prefix cache. Design for stable prefixes where possible.


# Conversation Growth: Cache Behavior Over Multi-Turn

In a real agent loop, the conversation grows turn by turn. With prefix caching,
each new turn only needs KV computation for the **new tokens** — the entire prefix
(system prompt + all previous turns) is cached.

But when a conversation *forks* (e.g., the agent retries with a different tool call),
the fork point invalidates the suffix cache. This is the trade-off between
"linear conversation" (cache-friendly) and "tree-of-thought" (cache-hostile).

In [ ]:
# ── Simulate a growing conversation (linear) ─────────────────────────
conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

turn_pairs = [
    ("My order #ORD-8842 hasn't arrived.", "I'll look into that for you."),
    ("It's been three weeks now.", "I understand your frustration. Let me check the status."),
    ("Can you expedite the shipping?", "I'll escalate this to our logistics team."),
    ("What about a refund if it doesn't arrive by Friday?", "Our policy allows refunds after 21 business days."),
    ("Okay, can you also check order #ORD-9901?", "Sure, let me pull up that order as well."),
    ("That one has the wrong item. I got a red widget instead of blue.", "I'm sorry about that. Let me initiate a return for the incorrect item."),
]

print("=" * 70)
print(" MULTI-TURN CONVERSATION — LINEAR GROWTH (cache-friendly)")
print("=" * 70)

linear_results = []
for i, (user_msg, assistant_msg) in enumerate(turn_pairs):
    conversation.append({"role": "user", "content": user_msg})
    r = timed_completion(local_client, conversation, max_tokens=128)
    linear_results.append({"turn": i + 1, **r})
    # Add the assistant response to keep the conversation going
    conversation.append({"role": "assistant", "content": assistant_msg})
    print(
        f"  Turn {i+1} │ {r['prompt_tokens']:>5} prompt tok │ "
        f"{r['latency_s']:>6.3f}s │ {r['tokens_per_sec']:>6.1f} tok/s"
    )

# ── Now fork: same base conversation, different last message ─────────
print(f"\n{'─' * 70}")
print(" CONVERSATION FORK — cache invalidation at the branch point")
print("─" * 70)

# Fork A: continue the original conversation
fork_a = conversation.copy()
fork_a.append({"role": "user", "content": "Actually, forget the return. Just send me a replacement."})

# Fork B: different branch from the same point
fork_b = conversation.copy()
fork_b.append({"role": "user", "content": "I want to speak with a supervisor about both orders."})

r_a = timed_completion(local_client, fork_a, max_tokens=128)
r_b = timed_completion(local_client, fork_b, max_tokens=128)

print(f"  Fork A (replacement): {r_a['latency_s']:>6.3f}s │ {r_a['prompt_tokens']} prompt tok")
print(f"  Fork B (supervisor):  {r_b['latency_s']:>6.3f}s │ {r_b['prompt_tokens']} prompt tok")
print(f"\n  → Both forks share the same prefix and differ only in the last message.")
print(f"    With prefix caching, the shared history is not recomputed.")

 MULTI-TURN CONVERSATION — LINEAR GROWTH (cache-friendly)
  Turn 1 │    51 prompt tok │  0.281s │   35.6 tok/s
  Turn 2 │    76 prompt tok │  0.279s │   35.8 tok/s
  Turn 3 │   104 prompt tok │  0.525s │   38.1 tok/s
  Turn 4 │   135 prompt tok │  0.661s │   37.8 tok/s
  Turn 5 │   171 prompt tok │  0.502s │   37.9 tok/s
  Turn 6 │   208 prompt tok │  0.272s │   33.1 tok/s

──────────────────────────────────────────────────────────────────────
 CONVERSATION FORK — cache invalidation at the branch point
──────────────────────────────────────────────────────────────────────
  Fork A (replacement):  0.343s │ 246 prompt tok
  Fork B (supervisor):   0.336s │ 245 prompt tok

  → Both forks share the same prefix and differ only in the last message.
    With prefix caching, the shared history is not recomputed.


# Backend Portability: Same Client, Different Server

A key deployment principle: **your agent code should be backend-agnostic.** Since all
three backends expose the OpenAI-compatible API, switching is a config change, not a
code change. Below we show how the same `OpenAI` client constructor works for each.

In [ ]:
# ── Backend configurations — same OpenAI client, different base_url ───
#
# Each backend uses its own entrypoint / CLI, but they all expose
# an OpenAI-compatible /v1/chat/completions endpoint.
#
# vLLM engine args reference:
#   https://docs.vllm.ai/en/stable/configuration/engine_args/

MODEL_EXAMPLE = "Qwen/Qwen2.5-3B-Instruct"

BACKEND_CONFIGS = {
    "vLLM": {
        "base_url": "http://localhost:8001/v1",
        "start_cmd": [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            f"--host=0.0.0.0",
            f"--port=8001",
            f"--model={MODEL_EXAMPLE}",
            f"--served-model-name=qwen",
            f"--tensor-parallel-size=1",
            f"--gpu-memory-utilization=0.90",
            f"--max-model-len=4096",
            f"--dtype=auto",
            f"--swap-space=4",
            f"--max-num-seqs=64",
            "--enforce-eager",
            "--enable-prefix-caching",
            "--disable-log-stats",
        ],
        "docs": "https://docs.vllm.ai/en/stable/configuration/engine_args/",
    },
    "TGI": {
        "base_url": "http://localhost:8002/v1",
        "start_cmd": [
            "text-generation-launcher",
            f"--model-id={MODEL_EXAMPLE}",
            "--port=8002",
            "--hostname=0.0.0.0",
            "--max-input-tokens=4096",
            "--max-total-tokens=4608",
            "--dtype=float16",
        ],
        "docs": "https://huggingface.co/docs/text-generation-inference",
    },
    "SGLang": {
        "base_url": "http://localhost:8003/v1",
        "start_cmd": [
            "python", "-m", "sglang.launch_server",
            f"--model-path={MODEL_EXAMPLE}",
            f"--served-model-name=qwen",
            "--port=8003",
            "--host=0.0.0.0",
        ],
        "docs": "https://docs.sglang.ai/",
    },
}

print("=" * 70)
print(" BACKEND PORTABILITY — YOUR AGENT CODE DOESN'T CHANGE")
print("=" * 70)

for name, cfg in BACKEND_CONFIGS.items():
    cmd_str = " \\\n              ".join(cfg["start_cmd"])
    print(f"\n  ── {name} ──")
    print(f"  Docs:     {cfg['docs']}")
    print(f"  Start:    {cmd_str}")
    print(f"  Client:   OpenAI(base_url='{cfg['base_url']}', api_key='not-needed')")
    print(f"  Call:     client.chat.completions.create(model='qwen', messages=...)")

print(f"\n{'─' * 70}")
print("  → Same chat.completions.create() call. Swap the base_url and you've")
print("    switched your entire inference backend. No agent code changes.")
print("    This is why using the OpenAI SDK as your client library matters —")
print("    it's the lingua franca of inference APIs.")

 BACKEND PORTABILITY — YOUR AGENT CODE DOESN'T CHANGE

  ── vLLM ──
  Docs:     https://docs.vllm.ai/en/stable/configuration/engine_args/
  Start:    python \
              -m \
              vllm.entrypoints.openai.api_server \
              --host=0.0.0.0 \
              --port=8001 \
              --model=Qwen/Qwen2.5-3B-Instruct \
              --served-model-name=qwen \
              --tensor-parallel-size=1 \
              --gpu-memory-utilization=0.90 \
              --max-model-len=4096 \
              --dtype=auto \
              --swap-space=4 \
              --max-num-seqs=64 \
              --enforce-eager \
              --enable-prefix-caching \
              --disable-log-stats
  Client:   OpenAI(base_url='http://localhost:8001/v1', api_key='not-needed')
  Call:     client.chat.completions.create(model='qwen', messages=...)

  ── TGI ──
  Docs:     https://huggingface.co/docs/text-generation-inference
  Start:    text-generation-launcher \
              --model-id=Qwen/Q

## Summary Dashboard

In [ ]:
print("=" * 70)
print(" DEPLOYMENT PHYSICS — SUMMARY")
print("=" * 70)

print(f"""
  ┌─────────────────────────────────────────────────────────────────┐
  │  COLD START                                                     │
  │  Server boot:            {cold_start_seconds:>6.1f}s                              │
  │  1st request:            {cold_req:>6.3f}s                              │
  │  Total (boot + 1st req): {total_to_first:>6.1f}s                              │
  │  Warm request avg:       {warm_avg:>6.3f}s                              │
  │  Cold/Warm ratio:        {total_to_first / warm_avg:>5.0f}×                               │
  │                                                                 │
  │  CONTEXT LENGTH                                                 │
  │  0 turns  → {context_results[0]['latency_s']:>6.3f}s                                      │
  │  {turn_counts[-1]} turns → {context_results[-1]['latency_s']:>6.3f}s  ({context_results[-1]['latency_s'] / context_results[0]['latency_s']:.1f}× slower)                        │
  │                                                                 │
  │  PREFIX CACHING (A/B)                                           │
  │  Without (avg Q2-5): {avg_no_2_5:>6.3f}s                                 │
  │  With    (avg Q2-5): {avg_with_2_5:>6.3f}s  ({cached_speedup:.2f}× faster)              │
  │                                                                 │
  │  CACHE INVALIDATION                                             │
  │  Same prefix (return):   {invalidation_results['back_to_support']['latency_s']:>6.3f}s (cache hit)               │
  │  New prefix (switch):    {invalidation_results['returns_specialist']['latency_s']:>6.3f}s (cache miss)              │
  │                                                                 │
  │  MODEL:  {VLLM_MODEL:<40}            │
  └─────────────────────────────────────────────────────────────────┘
""")

 DEPLOYMENT PHYSICS — SUMMARY

  ┌─────────────────────────────────────────────────────────────────┐
  │  COLD START                                                     │
  │  Server boot:             124.1s                              │
  │  1st request:             0.458s                              │
  │  Total (boot + 1st req):  124.6s                              │
  │  Warm request avg:        0.277s                              │
  │  Cold/Warm ratio:          449×                               │
  │                                                                 │
  │  CONTEXT LENGTH                                                 │
  │  0 turns  →  0.282s                                      │
  │  30 turns →  0.309s  (1.1× slower)                        │
  │                                                                 │
  │  PREFIX CACHING (A/B)                                           │
  │  Without (avg Q2-5):  0.450s                                 │
  │  With    (a

# Cleanup

In [ ]:
# ── Stop the vLLM server ──────────────────────────────────────────────
stop_vllm_server(vllm_proc)

# ── Clean up log file ─────────────────────────────────────────────────
if os.path.exists(VLLM_LOG):
    os.remove(VLLM_LOG)
    print(f"  Removed {VLLM_LOG}")

print("  Done.")

  Server stopped (PID 4853)
  Removed vllm_server.log
  Done.
